<a href="https://colab.research.google.com/github/JAVERIAADIL/Learning-GPU-infrastructure/blob/module1/Multiheaded_with_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
import torch
import torch.nn as nn
import math
import time

In [26]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Running on: {device}")
#basically it checks if the gpu is available on the machine and assign cuda or cpu to device so we can tell pytorch where to run our tensors
# and tensors and same as numpy mutidimensional array but they additiinally knows which device they live on gpu or cpu and can track gradient for back propogation


Running on: cuda


In [27]:
class MultiheadedAttention():
  def __init__ (self, d_model, num_of_heads, input):
    self.d_model = d_model
    self.num_of_heads = num_of_heads
    self.input = input
    self.d_k = (self.d_model//self.num_of_heads)
    #here input is defined by torch and for q k v we donot manullay define wq1 wq2 etc
    # we use nn.linear whose formula is xA.T +b  where x is input and A is weight metrics define by torch on its own and multiply it with x so we need to transpose it and b is bias vector which is optional
    # earlier we are using weights on own so we are not transposing it
    # nn.linear(input feature, output features ) means if we have 2 rows and 4 columns in input which means 2 words define in 4 dimension or feature so in nn.linear input feature is 4
    # but for output feture unlike manual attention with numpy we first get q1 q2 etc then we calculate then concatenate here first we only get Q, K, V then we split it into 2 layers of each then calculate then concatenate
    # so output feature must be same as input input feature because in manual attention after concatenate we get same shape as input so here we consider it as single headed attention and take same shape of q k v as input so nn.linaer (4, 4)
    self.wq = nn.Linear(d_model, d_model).to(device) # here we are making the layer for each q k v then we apply this layer in our input to get q k v
    self.wk = nn.Linear(d_model, d_model).to(device) #for cuda we need to copy them to gpu
    self.wv = nn.Linear(d_model, d_model).to(device)
    self.wo = nn.Linear(d_model, d_model).to(device) #w0 metrics that we need to multiply after concatenation of heads so here after final output we apply this layer to output which automatically multiply weight metrics to it
  def split_heads(self):
     Q = self.wq(self.input)
     K = self.wk(self.input)
     V = self.wv(self.input)
     # now for split head we need to reshape or view never copies data always shares original memory while reshape shares memory if possible and copies data if necessary
     # in rehsape we do it q.reshape(inputs like how many words how many rows , num of heads in how many heads we need to view the features like 2, d_k means each head see how many feature if head is 2 and features are 4 so each head see 2 different feature from each other)
     #same for view q.view(input feature, num of heads, d_k)
     #but reshape is more flexible while view throw error if tensor is non contiguous
     #A non-contiguous tensor means its elements are not stored in an unbroken, sequential block of memory
     self.seq_len = self.input.shape[0] # it gives row values
     Q = Q.reshape(self.seq_len, self.num_of_heads, self.d_k)
     K = K.reshape(self.seq_len, self.num_of_heads, self.d_k)
     V = V.reshape(self.seq_len, self.num_of_heads, self.d_k)
     #for pytorch :any operation which we want to perform first is batch. A batch is just a group of independent items processed simultaneously in one operation .PyTorch always treats the first dimension as the batch, meaning it runs the math independently for each item in that dimension.
     #so we have to perform transpose. Before transpose, shape is (seq_len, num_heads, d_k) .PyTorch treats seq_len as the batch, so it processes each token independently but mixes the heads together inside each token's computation
     #After transpose, shape is (num_heads, seq_len, d_k) . PyTorch treats num_heads as the batch, so it processes each head independently, giving each head its own separate (seq_len, d_k) block to compute attention on without interference from other heads.
     Q = Q.transpose(0,1)# it means swap 1st and 2nd position
     K = K.transpose(0,1)
     V = V.transpose(0,1)
     return Q, K, V



  def attention(self, Q, K,V):
    #here transpose (-2, -1 )means swap last 2 positions  because now K is 3d not 2d so we donot directly use .T
    scores= (Q @ K.transpose(-2, -1))/torch.sqrt(torch.tensor(self.d_k, dtype=torch.float32)) #torch.tensor(self.d_k, dtype=torch.float32) converts d_k (e.g., 4) into a single floating-point scalar tensor (tensor(4.)), and torch.sqrt(...) computes its square root (tensor(2.)); we convert it to a tensor because torch.sqrt operates on PyTorch tensors and the attention formula requires dividing by √d_k. so we donot directly pass d_k
    output = torch.softmax(scores, dim = -1 ) @ V #dim= -1 as on axis we need to have values all equals to 1
    output = output.transpose(0,1) #it means swap 1st and 2nd position because we have to concatenate head and get original shape as input
    output = output.reshape(self.seq_len, self.d_model) #same as input shape
    output = self.wo(output) #* of w0
    #we have to store in output everytime because pytorch cannot modify inplace they make a new tensor
    return scores, output
# we need to have a single function that is calling all other function in class this is the entry point
  def output_function(self, verbose = False):
    #here adding verbose means it just act like a flag if it is true then we only print these statements otherwise not because if we printing thes for like 100 times in loop while calling it it causes more choas
    if verbose:
        print(f"Memory before: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
    Q, K, V = self.split_heads()
    if verbose:
        print(f"Memory after split: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
    scores, output = self.attention(Q, K, V)
    if verbose:
        print(f"Memory after attention: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
    return scores, output


In [28]:
input = torch.randn(8,4).to(device) #torch.randn(seq_len, d_model)
d_model = input.shape[1]
num_of_heads = 2

In [29]:
# only tensors and model weights live in memory so they need .to(device) ,plain Python integers like d_model and num_of_heads are just numbers in CPU RAM and have no concept of device
mmhh = MultiheadedAttention(d_model, num_of_heads, input)
# mmhh.to(device)  .to(device) only works on classes that inherit from nn.Module(which we learn later). our class doesn't, so it has no .to() method. So for now in init where we are defining linear layer we directly copy these to device
scores, output = mmhh.output_function(verbose=True)
print(f"Score: {scores}")
print(f"Output: {output}")
print(f"Score Shape: {scores.shape}")
print(f"Output Shape: {output.shape}")
for _ in range(3):
    _ = mmhh.output_function()

torch.cuda.synchronize()  # Wait for GPU to finish
start = time.time()
for _ in range(100):
    out = mmhh.output_function()
torch.cuda.synchronize()
end = time.time()
print(f"Custom Mutiheaded Attention: {(end-start)/100 * 1000:.3f} ms per call")

Memory before: 9.14 MB
Memory after split: 9.14 MB
Memory after attention: 9.15 MB
Score: tensor([[[ 1.9744e-01,  8.9304e-02,  3.7043e-02,  1.9915e-01,  1.8781e-01,
          -1.5391e-01,  1.0558e-01,  1.3636e-01],
         [-9.1744e-02, -3.9881e-02, -4.2107e-02, -1.5830e-01, -1.7895e-01,
           1.3238e-01, -1.1291e-01, -8.9149e-02],
         [-1.2640e+00, -6.0823e-01,  3.2505e-01,  2.1017e-01,  8.6814e-01,
          -3.8921e-01,  7.6606e-01, -2.9055e-01],
         [-1.5785e+00, -7.6174e-01,  4.3933e-01,  3.5069e-01,  1.2071e+00,
          -5.6770e-01,  1.0423e+00, -3.2825e-01],
         [-1.4516e+00, -6.8859e-01,  2.2080e-01, -1.6145e-01,  4.3539e-01,
          -7.4143e-02,  4.8862e-01, -4.9164e-01],
         [ 2.5313e-01,  1.2732e-01, -1.5003e-01, -2.6646e-01, -4.8665e-01,
           2.8560e-01, -3.7126e-01, -2.9802e-02],
         [-1.4460e+00, -6.8611e-01,  2.2236e-01, -1.5448e-01,  4.4258e-01,
          -7.9741e-02,  4.9292e-01, -4.8726e-01],
         [-1.2531e+00, -6.2022e-01,

In [30]:
official = nn.MultiheadAttention(embed_dim=d_model, num_heads=num_of_heads, batch_first=True ) # batch = true means the input shape is expected as (batch, seq_len, d_model) instead of the default (seq_len, batch, d_model).
official.to(device)
output, attn_weights = official(input, input, input) #attention weights are same as tensor after applying softmax and before multiply it with V it is official function which first return output then attention weights
print(f"Attention Weights: {attn_weights}")
print(f"Attention Weights: {attn_weights.shape}")
print(f"Final Output Of Official Multiheaded Attention: {output}")
print(f"Shape Of Official Multiheaded Attention: {output.shape}")
for _ in range(3):
    _ = official(input, input, input) # official multiheaded require q , k, v so we give input 3 times. self-attention means Q, K, V all come from the same source so we pass input 3 times ,the layer internally projects them into separate Q, K, V using its own weights

torch.cuda.synchronize()  # Wait for GPU to finish
start = time.time()
for _ in range(100):
    out = official(input, input, input)
torch.cuda.synchronize()
end = time.time()
print(f"Official Mutiheaded Attention: {(end-start)/100 * 1000:.3f} ms per call")

Attention Weights: tensor([[0.2067, 0.1531, 0.0923, 0.1075, 0.1168, 0.1158, 0.0986, 0.1092],
        [0.1365, 0.1290, 0.1210, 0.1276, 0.1197, 0.1191, 0.1196, 0.1275],
        [0.1179, 0.1215, 0.1221, 0.1153, 0.1422, 0.1329, 0.1301, 0.1181],
        [0.0840, 0.1052, 0.1488, 0.1275, 0.1295, 0.1360, 0.1439, 0.1251],
        [0.0637, 0.0855, 0.1819, 0.1605, 0.1020, 0.1081, 0.1476, 0.1507],
        [0.2321, 0.1509, 0.0821, 0.0981, 0.1307, 0.1066, 0.0953, 0.1044],
        [0.0642, 0.0876, 0.1752, 0.1493, 0.1140, 0.1163, 0.1511, 0.1424],
        [0.2297, 0.1559, 0.0740, 0.0809, 0.1474, 0.1291, 0.0951, 0.0879]],
       device='cuda:0', grad_fn=<SqueezeBackward1>)
Attention Weights: torch.Size([8, 8])
Final Output Of Official Multiheaded Attention: tensor([[ 0.1636, -0.2467, -0.0931,  0.2676],
        [ 0.2315, -0.3210, -0.1758,  0.3225],
        [ 0.2307, -0.3750, -0.1525,  0.3819],
        [ 0.2742, -0.4036, -0.2017,  0.3955],
        [ 0.3180, -0.3934, -0.3151,  0.3844],
        [ 0.1400, -0

In [31]:
# timing varies between runs due to GPU scheduling, memory caching, thermal throttling, and Python overhead
# this is why we average over 100 runs , but even then small differences between runs are normal like 0.4 or 0.5 for official one
# what matters is order of magnitude , if both are in same ballpark the implementation is correct